# Structural Damage Classification

## Model Comparison dan Feature Engineering


**Metrik utama:** Macro F1.  
**Model kandidat final:** LightGBM multiclass dengan `building_id` Prefix8 sebagai fitur kategorikal.  
**Submission terdahulu terbaik:** `0.57239` pada leaderboard publik menggunakan Prefix8 Original.

## 1. Import library


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "scripts").is_dir():
    raise FileNotFoundError("Jalankan notebook dari folder project atau folder notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "raw"
for filename in ("train.csv", "test.csv", "sample_submission.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"File belum ditemukan: {DATA_DIR / filename}")

In [2]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score,accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay)

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import lightgbm as lgb

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", __import__("sklearn").__version__)
print("LightGBM:", lgb.__version__)
from IPython.display import display
from scripts.preprocess import build_classical_preprocessor, prepare_lgb_categories, prepare_catboost_features, validate_competition_data
from scripts.feature_engineering import building_id_prefix, audit_prefix_groups
from scripts.train import train_final_lightgbm, make_catboost_pipeline, make_lgb_experiment_pipeline

pandas: 2.2.3
numpy: 2.3.5
scikit-learn: 1.8.0
LightGBM: 4.6.0


In [3]:
RUN_BASELINES = False
RUN_CLASSICAL_TUNING = False
RUN_CATBOOST = False
RUN_LGBM_EXPERIMENTS = False
RUN_FINAL_TRAINING = True

if RUN_CLASSICAL_TUNING and not RUN_BASELINES:
    raise ValueError("Aktifkan RUN_BASELINES untuk menjalankan tuning klasik.")

## 2. Load data

In [4]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())
validate_competition_data(train, test, sample_submission)

Train shape: (533465, 17)
Test shape: (228629, 16)
Sample submission shape: (228629, 2)


,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,damage_grade,superstructure
0,280803013271,28,2802,280202,3,42,504,21,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Attached-2 side,Rectangular,Grade 5,has_superstructure_mud_mortar_stone
1,311908002031,31,3104,310416,2,8,380,16,Flat,Cement-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,Timber-Planck,Not attached,Rectangular,Grade 1,has_superstructure_cement_mortar_brick
2,310303000362,31,3111,311106,2,37,375,16,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,Grade 5,has_superstructure_mud_mortar_stone
3,280806040201,28,2802,280206,2,70,319,15,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Attached-1 side,Rectangular,Grade 3,has_superstructure_adobe_mud
4,291301000371,29,2904,290404,1,25,495,6,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Brick/Stone,Not applicable,Not attached,Rectangular,Grade 4,has_superstructure_mud_mortar_stone


,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,superstructure
0,122207000141,12,1205,120504,1,8,240,10,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,has_superstructure_mud_mortar_stone
1,284909000871,28,2810,281006,3,4,384,22,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
2,241904000231,24,2409,240909,2,999,384,21,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
3,248308000891,24,2412,241202,2,16,527,19,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
4,300405002981,30,3001,300104,2,6,280,16,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Timber-Planck,Not attached,Rectangular,has_superstructure_mud_mortar_stone


,building_id,damage_grade
0,122207000141,Grade 3
1,284909000871,Grade 1
2,241904000231,Grade 5
3,248308000891,Grade 4
4,300405002981,Grade 2


## 5. EDA fitur dan penetapan peran kolom

In [5]:
target = "damage_grade"
id_col = "building_id"
random_state = 42
label_order = ["Grade 1", "Grade 2", "Grade 3", "Grade 4", "Grade 5"]
grade_to_int = {label: idx for idx, label in enumerate(label_order)}
int_to_grade = np.array(label_order)

numeric_cols = ["count_floors_pre_eq","age_building","plinth_area_sq_ft","height_ft_pre_eq"]

categorical_cols = ["district_id","vdcmun_id","ward_id","land_surface_condition","foundation_type",
    "roof_type","ground_floor_type","other_floor_type","position","plan_configuration","superstructure",]

feature_cols = [col for col in train.columns if col not in [id_col, target]]

print("Jumlah fitur numerik:", len(numeric_cols))
print("Jumlah fitur kategorikal:", len(categorical_cols))
print("Jumlah fitur model tanpa ID:", len(feature_cols))
print("Label order:", label_order)

assert set(numeric_cols + categorical_cols) == set(feature_cols) # make sure kolom fitur suda lengkap
assert train[target].notna().all() # make sure target tidak ada missing value
assert set(train[target].unique()) == set(label_order) # make sure label target sesuai
assert train[id_col].is_unique and test[id_col].is_unique # maake sure id tidak duplikat
assert set(feature_cols) == (set(test.columns) - {id_col}) # make sure fitur dan kolom dataset sesuai
print("Pemeriksaan skema passed.")

print("\nRingkasan numerik:")
display(train[numeric_cols].describe().T)
print("\nRingkasan kategorikal:")
display(train[categorical_cols].describe().T)

Jumlah fitur numerik: 4
Jumlah fitur kategorikal: 11
Jumlah fitur model tanpa ID: 15
Label order: ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4', 'Grade 5']
Pemeriksaan skema passed.

Ringkasan numerik:


,count,mean,std,min,25%,50%,75%,max
count_floors_pre_eq,533465.0,2.088000,0.655001,1.0,2.0,2.0,2.0,9.0
age_building,533465.0,24.337319,64.999055,0.0,9.0,16.0,27.0,999.0
plinth_area_sq_ft,533465.0,406.523453,226.610477,70.0,280.0,358.0,470.0,5000.0
height_ft_pre_eq,533465.0,16.047675,5.490926,6.0,12.0,16.0,18.0,99.0



Ringkasan kategorikal:


,count,mean,std,min,25%,50%,75%,max
district_id,533465.0,25.764134,5.812230,12.0,22.0,24.0,30.0,36.0
vdcmun_id,533465.0,2582.312433,581.640567,1201.0,2204.0,2410.0,3010.0,3611.0
ward_id,533465.0,258236.686092,58164.132189,120101.0,220401.0,241004.0,301005.0,361108.0


## 8. Validation split dan data preparation

In [6]:
x = train[feature_cols].copy()
y = train[target].copy()
x_test_raw = test[feature_cols].copy()

x_train, x_valid, y_train, y_valid = train_test_split(x,y,test_size=0.20,random_state=random_state,stratify=y,)

y_train_int = y_train.map(grade_to_int).astype(int)
y_valid_int = y_valid.map(grade_to_int).astype(int)

print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

split_summary = pd.DataFrame({
    "train_proportion": y_train.value_counts(normalize=True).reindex(label_order),
    "validation_proportion": y_valid.value_counts(normalize=True).reindex(label_order),})
display(split_summary.round(4))

x_train: (426772, 15)
x_valid: (106693, 15)
y_train: (426772,)
y_valid: (106693,)


,train_proportion,validation_proportion
damage_grade,,
Grade 1,0.1034,0.1034
Grade 2,0.1145,0.1145
Grade 3,0.1790,0.1790
Grade 4,0.2412,0.2412
Grade 5,0.3619,0.3619


## 9. Baseline scikit-learn: Pipeline dan ColumnTransformer


In [7]:
preprocessor = build_classical_preprocessor(numeric_cols, categorical_cols)

### Metode yang dibandingkan

- **Dummy:** batas minimum yang hanya memprediksi kelas mayoritas.
- **Logistic Regression:** baseline linear; versi `balanced` dicoba karena distribusi target tidak seimbang.
- **Linear SVM:** pembanding berbasis margin.
- **Decision Tree:** melihat kemampuan tree tunggal menangkap aturan nonlinier.
- **Random Forest:** menggabungkan banyak tree untuk mengurangi varians.

Dipakai konfigurasi terbaik yang pernah ditemukan lewat tuning terbatas untuk Decision Tree dan Random Forest.

In [ ]:
if RUN_BASELINES:
    def evaluate_sklearn_model(name, family, estimator):

        start = time.perf_counter()
        estimator.fit(x_train, y_train)
        pred = estimator.predict(x_valid)
        elapsed = time.perf_counter() - start

        result = {
            "model": name,
            "family": family,
            "macro_f1": f1_score(y_valid, pred, average="macro"),
            "accuracy": accuracy_score(y_valid, pred),
            "train_minutes": elapsed / 60,
            "best_iteration": np.nan,}

        classical_predictions[name] = pred
        classical_models[name] = estimator
        classical_results.append(result)

        print(
            f"{name}: Macro F1={result['macro_f1']:.4f} | "
            f"Accuracy={result['accuracy']:.4f} | "
            f"Time={result['train_minutes']:.2f} min")
        return result


    def make_sklearn_pipeline(model):
        return Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", model),])


    classical_results = []
    classical_predictions = {}
    classical_models = {}

    classical_candidates = [
        ("Dummy Most Frequent","Dummy",
            DummyClassifier(strategy="most_frequent"),),
        ("Logistic Regression","Logistic Regression",
            make_sklearn_pipeline(
                LogisticRegression(C=1.0,max_iter=2000,random_state=random_state,)),),
        ("Logistic Regression Balanced","Logistic Regression",
            make_sklearn_pipeline(
                LogisticRegression(C=3.0,class_weight="balanced",max_iter=2000,random_state=random_state,)),),
        ("Linear SVM Balanced","Linear SVM",make_sklearn_pipeline(
                LinearSVC(C=1.0,class_weight="balanced",dual="auto",max_iter=5000,random_state=random_state,)),),
        ("Decision Tree","Decision Tree",
            make_sklearn_pipeline(
                DecisionTreeClassifier(max_depth=25,min_samples_leaf=50,random_state=random_state,)),),
        ("Random Forest Balanced","Random Forest",
            make_sklearn_pipeline(
                RandomForestClassifier(n_estimators=100,max_depth=30,min_samples_split=5,min_samples_leaf=2,
                    max_features="sqrt",class_weight="balanced_subsample",max_samples=0.8,random_state=random_state,
                    n_jobs=-1,)),),]

    for name, family, estimator in classical_candidates:
        evaluate_sklearn_model(name, family, estimator)

    classical_results_df = (
        pd.DataFrame(classical_results)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True))

    display(
        classical_results_df[
            ["model", "family", "macro_f1", "accuracy", "train_minutes"]
        ].round(4))
else:
    print("Baseline scikit-learn: eksperimen dilewati.")

Baseline scikit-learn: eksperimen dilewati.


## Tuning model klasik

In [ ]:

classical_tuning_history = pd.DataFrame([{'model': 'Logistic Regression Balanced C=3.0', 'macro_f1_history': 0.5076, 'accuracy_history': 0.5389}, {'model': 'Logistic Regression Balanced C=0.3', 'macro_f1_history': 0.5075, 'accuracy_history': 0.5392}, {'model': 'Logistic Regression Balanced C=0.1', 'macro_f1_history': 0.5071, 'accuracy_history': 0.5393}, {'model': 'Decision Tree depth=25, leaf=50', 'macro_f1_history': 0.4983, 'accuracy_history': 0.5479}, {'model': 'Random Forest depth=30, leaf=2', 'macro_f1_history': 0.49809, 'accuracy_history': 0.53695}, {'model': 'Random Forest depth=30, leaf=2, 250 trees', 'macro_f1_history': 0.49797, 'accuracy_history': 0.53709}, {'model': 'Decision Tree depth=25, leaf=100', 'macro_f1_history': 0.4909, 'accuracy_history': 0.5441}, {'model': 'Random Forest depth=30, leaf=5', 'macro_f1_history': 0.48353, 'accuracy_history': 0.52506}, {'model': 'Decision Tree depth=25, balanced', 'macro_f1_history': 0.4798, 'accuracy_history': 0.5062}, {'model': 'Random Forest depth=20, leaf=5', 'macro_f1_history': 0.45865, 'accuracy_history': 0.5065}, {'model': 'Decision Tree depth=15, leaf=100', 'macro_f1_history': 0.4529, 'accuracy_history': 0.5233}])

In [ ]:
if RUN_CLASSICAL_TUNING:
    classical_tuning_configs = [
        {
            "name": "Logistic Regression Balanced C=0.1",
            "family": "Logistic Regression",
            "params": {"C": 0.1, "class_weight": "balanced", "max_iter": 2000},},
        {
            "name": "Logistic Regression Balanced C=0.3",
            "family": "Logistic Regression",
            "params": {"C": 0.3, "class_weight": "balanced", "max_iter": 2000},},
        {
            "name": "Decision Tree depth=15, leaf=100",
            "family": "Decision Tree",
            "params": {"max_depth": 15, "min_samples_leaf": 100, "class_weight": None},},
        {
            "name": "Decision Tree depth=25, leaf=100",
            "family": "Decision Tree",
            "params": {"max_depth": 25, "min_samples_leaf": 100, "class_weight": None},},
        {
            "name": "Decision Tree depth=25, balanced",
            "family": "Decision Tree",
            "params": {"max_depth": 25, "min_samples_leaf": 100, "class_weight": "balanced"},},
        {
            "name": "Random Forest depth=20, leaf=5",
            "family": "Random Forest",
            "params": {
                "n_estimators": 100, "max_depth": 20,
                "min_samples_split": 10, "min_samples_leaf": 5,
                "max_features": "sqrt", "class_weight": "balanced_subsample",
                "max_samples": 0.8, "n_jobs": -1,},},
        {
            "name": "Random Forest depth=30, leaf=5",
            "family": "Random Forest",
            "params": {
                "n_estimators": 100, "max_depth": 30,
                "min_samples_split": 10, "min_samples_leaf": 5,
                "max_features": "sqrt", "class_weight": "balanced_subsample",
                "max_samples": 0.8, "n_jobs": -1,},},
        {
            "name": "Random Forest depth=30, leaf=2, 250 trees",
            "family": "Random Forest",
            "params": {
                "n_estimators": 250, "max_depth": 30,
                "min_samples_split": 5, "min_samples_leaf": 2,
                "max_features": "sqrt", "class_weight": "balanced_subsample",
                "max_samples": 0.8, "n_jobs": -1,},},]

    def build_classical_tuning_estimator(family, params):
        if family == "Logistic Regression":
            estimator = LogisticRegression(
                C=params["C"],
                class_weight=params["class_weight"],
                max_iter=params["max_iter"],
                random_state=random_state,)
        elif family == "Decision Tree":
            estimator = DecisionTreeClassifier(
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                class_weight=params["class_weight"],
                random_state=random_state,)
        elif family == "Random Forest":
            estimator_params = params.copy()
            estimator_params["n_jobs"] = -1
            estimator = RandomForestClassifier(
                **estimator_params,
                random_state=random_state,)
        else:
            raise ValueError(f"Family tidak dikenali: {family}")

        return make_sklearn_pipeline(estimator)


    # Gunakan kembali hasil baseline untuk konfigurasi yang identik agar tidak perlu melakukan training berulang.
    baseline_aliases = {
        "Logistic Regression Balanced C=3.0": "Logistic Regression Balanced",
        "Decision Tree depth=25, leaf=50": "Decision Tree",
        "Random Forest depth=30, leaf=2": "Random Forest Balanced",}

    classical_tuning_runs = []
    classical_tuning_models = {}
    classical_tuning_predictions = {}

    for tuning_name, baseline_name in baseline_aliases.items():
        baseline_result = next(
            result for result in classical_results
            if result["model"] == baseline_name)
        baseline_pred = classical_predictions[baseline_name]

        classical_tuning_runs.append({
            "model": tuning_name,
            "family": baseline_result["family"],
            "macro_f1_current": f1_score(
                y_valid, baseline_pred, average="macro"),
            "accuracy_current": accuracy_score(y_valid, baseline_pred),
            "train_minutes": baseline_result["train_minutes"],
            "execution": "Reuse baseline",
            "parameters": "Sama dengan konfigurasi baseline",})
        classical_tuning_models[tuning_name] = classical_models[baseline_name]
        classical_tuning_predictions[tuning_name] = baseline_pred

    for config in classical_tuning_configs:
        name = config["name"]
        family = config["family"]
        params = config["params"]

        model = build_classical_tuning_estimator(family, params)

        start = time.perf_counter()
        model.fit(x_train, y_train)
        pred = model.predict(x_valid)
        elapsed_minutes = (time.perf_counter() - start) / 60

        result = {
            "model": name,"family": family,
            "macro_f1_current": f1_score(y_valid, pred, average="macro"),
            "accuracy_current": accuracy_score(y_valid, pred),
            "train_minutes": elapsed_minutes,"execution": "Retrained","parameters": str(params),}

        classical_tuning_runs.append(result)
        classical_tuning_models[name] = model
        classical_tuning_predictions[name] = pred

        print(
            f"{name}: Macro F1={result['macro_f1_current']:.5f} | "
            f"Accuracy={result['accuracy_current']:.5f} | "
            f"Time={elapsed_minutes:.2f} min")

    classical_tuning_results_df = pd.DataFrame(classical_tuning_runs)

    classical_tuning_comparison = classical_tuning_history.merge(
        classical_tuning_results_df,on="model",how="left",validate="one_to_one",)

    classical_tuning_comparison["macro_f1_difference"] = (
        classical_tuning_comparison["macro_f1_current"]
        - classical_tuning_comparison["macro_f1_history"])

    classical_tuning_comparison = (
        classical_tuning_comparison
        .sort_values("macro_f1_current", ascending=False)
        .reset_index(drop=True))

    display(classical_tuning_comparison[
        ["model","macro_f1_history","macro_f1_current","macro_f1_difference","accuracy_history","accuracy_current","train_minutes","execution",]
    ].round(5))
else:
    print("Tuning klasik: eksperimen dilewati.")

Tuning klasik: eksperimen dilewati.


## 10. CatBoost: boosting dengan fitur kategorikal native

CatBoost dicoba karena dataset memiliki banyak fitur kategorikal dan hubungan yang mungkin nonlinier. CatBoost Original dibandingkan dengan `SqrtBalanced`. Selain kemampuan model, pembobotan kelas diuji karena distribusi target tidak seimbang. Kedua eksperimen memakai fitur dan parameter inti yang sama, perubahan utama adalah `auto_class_weights`.

In [11]:
x_train_cb, x_valid_cb = prepare_catboost_features(x_train, x_valid, categorical_cols)

print("CatBoost train shape:", x_train_cb.shape)
print("CatBoost validation shape:", x_valid_cb.shape)
print("Target encoding:", grade_to_int)

catboost_task_type = "GPU" 

CatBoost train shape: (426772, 15)
CatBoost validation shape: (106693, 15)
Target encoding: {'Grade 1': 0, 'Grade 2': 1, 'Grade 3': 2, 'Grade 4': 3, 'Grade 5': 4}


In [ ]:
def fit_catboost_model(name,depth=8,learning_rate=0.05,l2_leaf_reg=5,auto_class_weights=None,):

    params = {"iterations": 4000,"depth": depth,"learning_rate": learning_rate,"l2_leaf_reg": l2_leaf_reg,
        "loss_function": "MultiClass","eval_metric": "TotalF1:average=Macro","random_seed": random_state,
        "task_type": catboost_task_type,"allow_writing_files": False,"verbose": 500,}

    if auto_class_weights is not None:
        params["auto_class_weights"] = auto_class_weights

    model = make_catboost_pipeline(params, categorical_cols)

    start = time.perf_counter()
    model.fit(x_train,y_train_int,model__cat_features=categorical_cols,model__eval_set=(x_valid_cb, y_valid_int),
        model__early_stopping_rounds=200,model__use_best_model=True,model__verbose=500,)
    elapsed = (time.perf_counter() - start) / 60

    pred_int = model.predict(x_valid).reshape(-1).astype(int)
    pred_labels = int_to_grade[pred_int]

    return {
        "model": name,
        "family": "CatBoost",
        "macro_f1": f1_score(
            y_valid, pred_labels, average="macro"),
        "accuracy": accuracy_score(y_valid, pred_labels),
        "train_minutes": elapsed,
        "best_iteration": int(model.named_steps["model"].best_iteration_),
        "tree_count": int(model.named_steps["model"].tree_count_),
        "auto_class_weights": auto_class_weights or "None",
        "model_object": model,
        "predictions": pred_labels,}


In [ ]:
if RUN_CATBOOST:
    # Menjalankan konfig konfig yang bisa dicoba
    catboost_configs = [
        {"name": "CatBoost Original",},
        {"name": "CatBoost Tune 4","depth": 10,"learning_rate": 0.05,"l2_leaf_reg": 5,},
        {"name": "CatBoost SqrtBalanced","auto_class_weights": "SqrtBalanced",},
        {"name": "CatBoost Balanced","auto_class_weights": "Balanced",},]

    catboost_runs = [
        fit_catboost_model(**config)
        for config in catboost_configs]

    catboost_models = {
        result["model"]: result["model_object"]
        for result in catboost_runs}

    catboost_predictions = {
        result["model"]: result["predictions"]
        for result in catboost_runs}

    catboost_results = [
        {
            key: value
            for key, value in result.items()
            if key not in ["model_object", "predictions"]}
        for result in catboost_runs]

    catboost_results_df = (
        pd.DataFrame(catboost_results)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True))

    display(
        catboost_results_df[
            ["model", "macro_f1", "accuracy","best_iteration", "tree_count","train_minutes", "auto_class_weights",]
        ].round(5))

else:
    print("CatBoost GPU: eksperimen dilewati.")

CatBoost GPU: eksperimen dilewati.


### Eksperimen CatBoost lain yang sudah dicoba

CatBoost juga sempat diuji dengan beberapa depth, learning rate, regularisasi, class weighting,  Hasil historis berikut menunjukkan mengapa CatBoost tetap menjadi pembanding yang layak, tetapi kemudian digantikan metode lain sebagai kandidat terbaik.

In [14]:
catboost_tuning_history = pd.DataFrame([
    {"experiment": "CatBoost Original", "macro_f1": 0.5338, "accuracy": 0.5773},
    {"experiment": "CatBoost Tune 3 (depth=8, lr=0.05, l2=5)", "macro_f1": 0.5364, "accuracy": 0.5790},
    {"experiment": "CatBoost Tune 4 (depth=10, lr=0.05, l2=5)", "macro_f1": 0.5363, "accuracy": 0.5791},
    {"experiment": "CatBoost SqrtBalanced", "macro_f1": 0.5407, "accuracy": 0.5749},
    {"experiment": "CatBoost Balanced", "macro_f1": 0.5380, "accuracy": 0.5644},
    {"experiment": "CatBoost depth=10, l2=10 + adjustment", "macro_f1": 0.54419, "accuracy": 0.57646},
    {"experiment": "CatBoost MultiClassOneVsAll + adjustment", "macro_f1": 0.54375, "accuracy": 0.57625},
    {"experiment": "CatBoost Age+Height", "macro_f1": 0.5321, "accuracy": 0.5768},
    {"experiment": "CatBoost Age+Height+Location", "macro_f1": 0.5321, "accuracy": 0.5769},
])
display(catboost_tuning_history.sort_values("macro_f1", ascending=False).round(5))

,experiment,macro_f1,accuracy
5,"CatBoost depth=10, l2=10 + adjustment",0.54419,0.57646
6,CatBoost MultiClassOneVsAll + adjustment,0.54375,0.57625
3,CatBoost SqrtBalanced,0.54070,0.57490
4,CatBoost Balanced,0.53800,0.56440
1,"CatBoost Tune 3 (depth=8, lr=0.05, l2=5)",0.53640,0.57900
2,"CatBoost Tune 4 (depth=10, lr=0.05, l2=5)",0.53630,0.57910
0,CatBoost Original,0.53380,0.57730
7,CatBoost Age+Height,0.53210,0.57680
8,CatBoost Age+Height+Location,0.53210,0.57690


## 11. LightGBM baseline: alternatif gradient boosting

CatBoost sudah memberi hasil luamayan baik, tetapi dicoba dibandingkan dengan implementasi gradient boosting lain. LightGBM cocok diuji pada data tabular besar karena menyediakan native categorical handling dan training berbasis histogram.

In [ ]:
y_train_lgb = y_train.map(grade_to_int).astype(int)
y_valid_lgb = y_valid.map(grade_to_int).astype(int)


x_train_lgb, x_valid_lgb = prepare_lgb_categories(x_train, x_valid, categorical_cols)

lgb_params = {"objective": "multiclass","n_estimators": 3000,"learning_rate": 0.05,
    "num_leaves": 127,"max_depth": -1,"min_child_samples": 100,"subsample": 0.8,
    "subsample_freq": 1,"colsample_bytree": 0.9,"reg_lambda": 5,"class_weight": "balanced",
    "random_state": random_state,"n_jobs": -1,"verbosity": -1,}

print("LightGBM features:", x_train_lgb.shape[1])
print("Categorical features:", len(categorical_cols))

LightGBM features: 15
Categorical features: 11


In [ ]:
if RUN_LGBM_EXPERIMENTS:
    lgb_baseline = make_lgb_experiment_pipeline(lgb_params, categorical_cols)
    start = time.perf_counter()
    lgb_baseline.fit(
        x_train,
        y_train_lgb,
        model__categorical_feature=categorical_cols,
        model__eval_set=[(x_valid_lgb, y_valid_lgb)],
        model__eval_metric="multi_logloss",
        model__callbacks=[early_stopping(150), log_evaluation(250)],)
    lgb_baseline_minutes = (time.perf_counter() - start) / 60

    baseline_proba = lgb_baseline.predict_proba(
        x_valid, num_iteration=lgb_baseline.named_steps["model"].best_iteration_)
    baseline_pred = lgb_baseline.named_steps["model"].classes_[np.argmax(baseline_proba, axis=1)]

    multipliers = np.array([1.00, 1.05, 1.05, 1.05, 1.00])
    baseline_pred_adjusted = lgb_baseline.named_steps["model"].classes_[
        np.argmax(baseline_proba * multipliers, axis=1)]

    lgb_baseline_results = []
    for name, pred in [
        ("LightGBM Baseline Original", baseline_pred),
        ("LightGBM Baseline Adjusted", baseline_pred_adjusted),]:
        pred_labels = int_to_grade[pred.astype(int)]
        lgb_baseline_results.append({
            "model": name,
            "family": "LightGBM",
            "macro_f1": f1_score(y_valid, pred_labels, average="macro"),
            "accuracy": accuracy_score(y_valid, pred_labels),
            "train_minutes": lgb_baseline_minutes,
            "best_iteration": int(lgb_baseline.named_steps["model"].best_iteration_),})

    print(f"Training time: {lgb_baseline_minutes:.2f} minutes")
    print("Best iteration:", lgb_baseline.named_steps["model"].best_iteration_)
    display(pd.DataFrame(lgb_baseline_results).round(5))
else:
    print("LightGBM baseline: eksperimen dilewati.")

LightGBM baseline: eksperimen dilewati.


## Investigasi Lanjutan 1: Interaksi Lokasi dan Karakteristik Bangunan

Karena nnilai F1-Score yang cenderung tidak menunjukan perubahan drastis dari awal metode klasif hingga lightgbm original, ditinjau lebih lanjut faktor faktor dari variabel yang mungkin bisa dijadikan sumber untuk menemukan pola tersembunyi. EDA awal menunjukkan bahwa distribusi tingkat kerusakan berbeda menurut lokasi dan karakteristik bangunan. Hal ini memunculkan hipotesis bahwa hubungan antara suatu karakteristik struktural dan tingkat kerusakan mungkin berbeda antarwilayah.

Untuk menguji kemungkinan tersebut, kita mencoba dua fitur interaksi:

- **VDCmun × Foundation:** menguji apakah kombinasi wilayah VDC/Municipality dan jenis fondasi memberikan informasi tambahan.
- **District × Roof:** menguji apakah kombinasi distrik dan jenis atap memberikan informasi tambahan.

In [ ]:
if RUN_LGBM_EXPERIMENTS:
    interaction_multipliers = np.array([1.00, 1.05, 1.05, 1.05, 1.00])
    interaction_models = {}
    interaction_predictions = {}
    interaction_rows = []
    # Baseline dari model yang sudah dilatih sebelumnya.
    for adjustment, pred in [
        ("Original", baseline_pred),
        ("Adjusted", baseline_pred_adjusted),]:
        interaction_rows.append({
            "model": "LightGBM Baseline",
            "adjustment": adjustment,
            "macro_f1": f1_score(
                y_valid_lgb, pred, average="macro"),
            "accuracy": accuracy_score(y_valid_lgb, pred),
            "feature_count": x_train_lgb.shape[1],
            "best_iteration": int(lgb_baseline.named_steps["model"].best_iteration_),
            "train_minutes": lgb_baseline_minutes,})

    def run_lgb_interaction(left_col, right_col, feature_name, display_name):
        """Latih LightGBM dengan satu fitur interaksi dan nilai validation-nya."""
        x_train_new = x_train_lgb.copy()
        x_valid_new = x_valid_lgb.copy()
        train_pair = (
            x_train_new[left_col].astype("string").fillna("__MISSING__") + "__"
            + x_train_new[right_col].astype("string").fillna("__MISSING__"))
        valid_pair = (
            x_valid_new[left_col].astype("string").fillna("__MISSING__")+ "__"
            + x_valid_new[right_col].astype("string").fillna("__MISSING__"))

        # Kategori kombinasi dipelajari dari training
        pair_categories = pd.Index(train_pair.unique())
        x_train_new[feature_name] = pd.Categorical(
            train_pair.to_numpy(),
            categories=pair_categories,)
        x_valid_new[feature_name] = pd.Categorical(
            valid_pair.to_numpy(),
            categories=pair_categories,)

        interaction_cat_cols = categorical_cols + [feature_name]
        model = make_lgb_experiment_pipeline(lgb_params, interaction_cat_cols)

        start = time.perf_counter()
        model.fit(
            x_train_new, y_train_lgb,
            model__categorical_feature=interaction_cat_cols,
            model__eval_set=[(x_valid_new, y_valid_lgb)],
            model__eval_metric="multi_logloss",
            model__callbacks=[early_stopping(150), log_evaluation(250)],)
        elapsed = (time.perf_counter() - start) / 60

        proba = model.predict_proba(x_valid_new,num_iteration=model.named_steps["model"].best_iteration_,)

        pred_original = model.named_steps["model"].classes_[np.argmax(proba, axis=1)]

        pred_adjusted = model.named_steps["model"].classes_[np.argmax(proba * interaction_multipliers, axis=1)]

        interaction_models[feature_name] = model
        interaction_predictions[feature_name] = {"original": pred_original,"adjusted": pred_adjusted,}

        print(f"\n{display_name}")
        print("Jumlah kombinasi training:", len(pair_categories))
        print("Validation combination coverage:",round(valid_pair.isin(pair_categories).mean(), 4))
        print("Best iteration:", model.named_steps["model"].best_iteration_)
        print("Training time:", round(elapsed, 2), "minutes")

        for adjustment, pred in [("Original", pred_original),("Adjusted", pred_adjusted),]:
            interaction_rows.append({
                "model": display_name,
                "adjustment": adjustment,
                "macro_f1": f1_score(y_valid_lgb, pred, average="macro"),
                "accuracy": accuracy_score(y_valid_lgb, pred),
                "feature_count": x_train_new.shape[1],
                "best_iteration": int(model.named_steps["model"].best_iteration_),
                "train_minutes": elapsed,})

    run_lgb_interaction("vdcmun_id","foundation_type","vdcmun_foundation","VDCmun × Foundation",)

    run_lgb_interaction("district_id","roof_type","district_roof_key","District × Roof",)

    interaction_results_df = (
        pd.DataFrame(interaction_rows)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True))

    display(interaction_results_df.round(5))

else:
    print("Interaksi lokasi: eksperimen dilewati.")

Interaksi lokasi: eksperimen dilewati.


## Investigasi Lanjutan 2: Apakah `building_id` Memiliki Struktur?

Interaksi lokasi dan karakteristik bangunan belum memberikan peningkatan yang konsisten. Karena itu, diperiksa kemungkinan lain yang belum digunakan pada baseline, yaitu struktur identifier bangunan.

`building_id` awalnya dikeluarkan dari fitur karena tidak tepat diperlakukan sebagai angka ordinal. Namun, setelah diamati lebih dalam, ditemukan kecocokan dua digit awal dengan `district_id`, ukuran kelompok dari prefix yang berbeda panjangnya, variasi proporsi antar kelompok, serta cakupan prefix validation dan test yang sudah muncul pada training split.

Hasil audit digunakan untuk merumuskan kandidat fitur. Variasi antar kelompok merupakan petunjuk untuk diuji, bukan bukti bahwa seluruh digit ID merupakan kode geografis resmi atau bahwa prefix pasti meningkatkan prediksi.



Kita membandingkan prefix 2, 4, 6, 8, dan 10 digit. 

In [ ]:
id_train_audit = (
    train.loc[x_train.index, id_col]
    .astype("string")
    .str.zfill(12)
    .reset_index(drop=True))
id_valid_audit = (
    train.loc[x_valid.index, id_col]
    .astype("string")
    .str.zfill(12)
    .reset_index(drop=True))
id_test_audit = (
    test[id_col]
    .astype("string")
    .str.zfill(12)
    .reset_index(drop=True))
district_train_audit = (
    train.loc[x_train.index, "district_id"]
    .reset_index(drop=True))
y_train_audit = y_train.reset_index(drop=True)

# Periksa kecocokan dua digit pertama dengan district_id
district_prefix_match = (
    id_train_audit.str[:2].astype(int).to_numpy()
    == district_train_audit.astype(int).to_numpy()).mean()

print("Baris data training yang dua digit pertama ID-nya sama dengan district_id:",round(float(district_prefix_match), 4))
prefix_audit = audit_prefix_groups(
    id_train_audit, y_train_audit, id_valid_audit, id_test_audit,
    label_order=label_order, prefix_lengths=[2, 4, 6, 8, 10],)
display(prefix_audit.round(4))

Baris data training yang dua digit pertama ID-nya sama dengan district_id: 1.0


,prefix_length,num_groups,median_group_size,share_groups_20plus,validation_prefix_coverage,test_prefix_coverage,grade_1_rate_std,grade_2_rate_std,grade_3_rate_std,grade_4_rate_std,grade_5_rate_std
0,2,11,42973.0,1.0000,1.0000,1.0000,0.1078,0.0781,0.0847,0.0784,0.2504
1,4,627,559.0,1.0000,1.0000,1.0000,0.1267,0.1082,0.1391,0.1585,0.2935
2,6,5666,59.0,0.9506,1.0000,1.0000,0.1489,0.1407,0.1791,0.2071,0.3228
3,8,6103,57.0,0.9254,0.9998,0.9998,0.1702,0.1455,0.1802,0.2110,0.3258
4,10,74571,6.0,0.0000,0.9969,0.9967,0.2217,0.1982,0.2377,0.2756,0.3702


### Interpretasi audit ID

Dua digit pertama `building_id` cocok dengan `district_id` pada seluruh baris train yang diperiksa. Prefix 4 digit membentuk kelompok besar; prefix 6 dan 8 membentuk lebih banyak kelompok yang sebagian besar masih memiliki puluhan anggota. Prefix 10 membentuk sangat banyak kelompok kecil sehingga berisiko terlalu spesifik.

Untuk menguji manfaatnya antar prefix, kita masukkan prefix sebagai **kategori**, lalu membandingkan Macro F1 pada validation yang sama.

## Investigasi Lanjutan 3: Membandingkan Panjang Prefix `building_id`

Audit ID menunjukkan bahwa panjang prefix memengaruhi jumlah dan ukuran kelompok. Prefix yang terlalu pendek mungkin kurang spesifik, sedangkan prefix yang terlalu panjang dapat membentuk kelompok kecil yang sulit dipelajari secara stabil.

Untuk menguji hal tersebut, kita membuat fitur kategorikal dari 6, 8, dan 10 digit pertama `building_id`. Semua model menggunakan konfigurasi LightGBM yang sama dengan baseline, validation split yang sama, serta Macro F1 sebagai metrik perbandingan utama.

In [ ]:
if RUN_LGBM_EXPERIMENTS:

    prefix_multipliers = np.array([1.00, 1.05, 1.05, 1.05, 1.00])

    prefix_models = {}
    prefix_features = {}
    prefix_predictions = {}
    prefix_probabilities = {}
    prefix_metadata = {}

    prefix_results_rows = []

    # Masukkan baseline yang sudah dilatih, tanpa training ulang.
    for adjustment, pred in [
        ("Original", baseline_pred),
        ("Adjusted", baseline_pred_adjusted), ]:
        prefix_results_rows.append({
            "model": "LightGBM Baseline",
            "prefix_length": 0,
            "adjustment": adjustment,
            "macro_f1": f1_score(
                y_valid_lgb, pred, average="macro"
            ),
            "accuracy": accuracy_score(y_valid_lgb, pred),
            "best_iteration": int(lgb_baseline.named_steps["model"].best_iteration_),
            "train_minutes": lgb_baseline_minutes,
            "validation_prefix_coverage": np.nan,
            "test_prefix_coverage": np.nan,})


    # Ambil ID dalam urutan baris yang sama dengan split yang digunakan.
    id_train_for_prefix = (
        train.loc[x_train.index, id_col]
        .astype("string")
        .str.zfill(12)
        .reset_index(drop=True))

    id_valid_for_prefix = (
        train.loc[x_valid.index, id_col]
        .astype("string")
        .str.zfill(12)
        .reset_index(drop=True))

    id_test_for_prefix = (
        test[id_col]
        .astype("string")
        .str.zfill(12)
        .reset_index(drop=True))


    for k in [6, 8, 10]:
        print(f"\nTraining LightGBM with Prefix{k}")

        prefix_col = f"building_id_prefix_{k}"

        train_prefix = id_train_for_prefix.str[:k]
        valid_prefix = id_valid_for_prefix.str[:k]
        test_prefix = id_test_for_prefix.str[:k]

        # Kategori hanya dipelajari dari training split.
        prefix_categories = pd.Index(train_prefix.unique())

        x_train_new = x_train_lgb.copy()
        x_valid_new = x_valid_lgb.copy()

        x_train_new[prefix_col] = pd.Categorical(
            train_prefix.to_numpy(),
            categories=prefix_categories,)

        x_valid_new[prefix_col] = pd.Categorical(
            valid_prefix.to_numpy(),
            categories=prefix_categories,)

        categorical_cols_new = categorical_cols + [prefix_col]

        validation_coverage = valid_prefix.isin(
            prefix_categories
        ).mean()

        test_coverage = test_prefix.isin(
            prefix_categories
        ).mean()

        print("Training groups:", len(prefix_categories))
        print("Validation prefix coverage:", round(validation_coverage, 4))
        print("Test prefix coverage:", round(test_coverage, 4))

        # Parameter sama dengan baseline LightGBM.
        model = make_lgb_experiment_pipeline(lgb_params, categorical_cols_new)

        start = time.perf_counter()

        model.fit(
            x_train_new,
            y_train_lgb,
            model__categorical_feature=categorical_cols_new,
            model__eval_set=[(x_valid_new, y_valid_lgb)],
            model__eval_metric="multi_logloss",
            model__callbacks=[early_stopping(150), log_evaluation(250)],)

        elapsed = (time.perf_counter() - start) / 60

        proba = model.predict_proba(
            x_valid_new,
            num_iteration=model.named_steps["model"].best_iteration_,)

        pred_original = model.named_steps["model"].classes_[
            np.argmax(proba, axis=1)]

        pred_adjusted = model.named_steps["model"].classes_[
            np.argmax(proba * prefix_multipliers, axis=1)]

        # Simpan model, fitur, probabilitas, dan prediksi untuk evaluasi berikutnya.
        prefix_models[k] = model

        prefix_features[k] = {
            "x_train": x_train_new,
            "x_valid": x_valid_new,
            "categorical_cols": categorical_cols_new,
            "prefix_categories": prefix_categories,}

        prefix_predictions[k] = {
            "original": pred_original,
            "adjusted": pred_adjusted,}

        prefix_probabilities[k] = proba

        prefix_metadata[k] = {
            "best_iteration": int(model.named_steps["model"].best_iteration_),
            "train_minutes": elapsed,
            "validation_prefix_coverage": validation_coverage,
            "test_prefix_coverage": test_coverage,}

        for adjustment, pred in [
            ("Original", pred_original),
            ("Adjusted", pred_adjusted),]:
            prefix_results_rows.append({
                "model": f"LightGBM Prefix{k}",
                "prefix_length": k,
                "adjustment": adjustment,
                "macro_f1": f1_score(y_valid_lgb, pred, average="macro"),
                "accuracy": accuracy_score(y_valid_lgb, pred),
                "best_iteration": int(model.named_steps["model"].best_iteration_),
                "train_minutes": elapsed,
                "validation_prefix_coverage": validation_coverage,
                "test_prefix_coverage": test_coverage,})

        print("Best iteration:", model.named_steps["model"].best_iteration_)
        print("Training time:", round(elapsed, 2), "minutes")


    prefix_results_df = (
        pd.DataFrame(prefix_results_rows)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True))

    display(prefix_results_df.round(5))

    lgb_prefix8 = prefix_models[8]

    x_train_p8 = prefix_features[8]["x_train"]
    x_valid_p8 = prefix_features[8]["x_valid"]
    categorical_cols_p8 = prefix_features[8]["categorical_cols"]

    prefix8_pred = prefix_predictions[8]["original"]
    prefix8_pred_adjusted = prefix_predictions[8]["adjusted"]
    prefix8_proba = prefix_probabilities[8]

    prefix8_minutes = prefix_metadata[8]["train_minutes"]

else:
    print("Prefix6/8/10: eksperimen dilewati.")

Prefix6/8/10: eksperimen dilewati.


## Investigasi Lanjutan 4: Verifikasi Prefix8 pada CatBoost untuk pembanding dengan LightGBM

CatBoost sebelumnya menjadi salah satu model dengan performa terbaik sebelum LightGBM diuji menggunakan Prefix8. Untuk mengetahui apakah penambahan fitur ini mengubah model terbaik, CatBoost SqrtBalanced diuji dengan dan tanpa building_id_prefix_8.

Hasilnya dibandingkan dengan LightGBM baseline dan LightGBM Prefix8 menggunakan validation split yang sama serta Macro F1 dan accuracy.

In [ ]:
if RUN_CATBOOST and RUN_LGBM_EXPERIMENTS:
    # Ambil CatBoost SqrtBalanced yang sudah dilatih sebelumnya.
    catboost_sqrt = catboost_models["CatBoost SqrtBalanced"]

    # Buat Prefix8 dengan urutan baris sesuai split training-validation.
    prefix8_train_cb = (
        train.loc[x_train.index, id_col]
        .astype("string")
        .str.zfill(12)
        .str[:8])

    prefix8_valid_cb = (
        train.loc[x_valid.index, id_col]
        .astype("string")
        .str.zfill(12)
        .str[:8])

    x_train_cb_p8 = x_train_cb.copy()
    x_valid_cb_p8 = x_valid_cb.copy()

    # CatBoost menerima fitur kategorikal sebagai string.
    x_train_cb_p8["building_id_prefix_8"] = (
        prefix8_train_cb.astype(str).to_numpy())

    x_valid_cb_p8["building_id_prefix_8"] = (
        prefix8_valid_cb.astype(str).to_numpy())

    categorical_cols_cb_p8 = (
        categorical_cols + ["building_id_prefix_8"])

    print("CatBoost original features:", x_train_cb.shape[1])
    print("CatBoost Prefix8 features:", x_train_cb_p8.shape[1])
    print("Validation Prefix8 coverage:",
        round(prefix8_valid_cb.isin(set(prefix8_train_cb)).mean(),4))

    # Dengan konfigurasi inti yang sama dengan CatBoost SqrtBalanced.
    # Iteration budget tetap 4000, seperti konfigurasi baseline CatBoost.
    catboost_p8_params = catboost_sqrt.named_steps["model"].get_params()

    catboost_p8_params.update({
        "iterations": 4000,
        "depth": 8,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5,
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "random_seed": random_state,
        "task_type": catboost_task_type,
        "auto_class_weights": "SqrtBalanced",
        "allow_writing_files": False,
        "verbose": 500,})

    catboost_sqrt_prefix8 = make_catboost_pipeline(
        catboost_p8_params, categorical_cols_cb_p8)

    start = time.perf_counter()

    catboost_sqrt_prefix8.fit(
        x_train_cb_p8,
        y_train_int,
        model__cat_features=categorical_cols_cb_p8,
        model__eval_set=(x_valid_cb_p8, y_valid_int),
        model__early_stopping_rounds=200,
        model__use_best_model=True,
        model__verbose=500,)

    catboost_prefix8_minutes = (
        time.perf_counter() - start) / 60

    print(
        "CatBoost Prefix8 training time:",
        round(catboost_prefix8_minutes, 2),
        "minutes")
    print("Best iteration:", catboost_sqrt_prefix8.named_steps["model"].best_iteration_)


    # CatBoost SqrtBalanced original, model yang sudah ada.
    catboost_original_pred = (
        catboost_sqrt
        .predict(x_valid_cb)
        .reshape(-1)
        .astype(int)
    )

    # CatBoost SqrtBalanced + Prefix8.
    catboost_prefix8_pred = (
        catboost_sqrt_prefix8
        .predict(x_valid_cb_p8)
        .reshape(-1)
        .astype(int)
    )

    # LightGBM baseline dan Prefix8, tanpa adjustment.
    lightgbm_baseline_pred = baseline_pred
    lightgbm_prefix8_pred = prefix_predictions[8]["original"]

    cross_model_predictions = [
        ("CatBoost SqrtBalanced Original",
         catboost_original_pred,
         x_train_cb.shape[1],
         catboost_sqrt.named_steps["model"].tree_count_),

        ("CatBoost SqrtBalanced + Prefix8",
         catboost_prefix8_pred,
         x_train_cb_p8.shape[1],
         catboost_sqrt_prefix8.named_steps["model"].tree_count_),

        ("LightGBM Baseline Original",
         lightgbm_baseline_pred,
         x_train_lgb.shape[1],
         lgb_baseline.named_steps["model"].best_iteration_),

        ("LightGBM Prefix8 Original",
         lightgbm_prefix8_pred,
         x_train_p8.shape[1],
         lgb_prefix8.named_steps["model"].best_iteration_),]

    cross_model_rows = []

    for name, pred, feature_count, tree_count in cross_model_predictions:
        cross_model_rows.append({
            "model": name,
            "feature_count": feature_count,
            "macro_f1": f1_score(
                y_valid_lgb, pred, average="macro"
            ),
            "accuracy": accuracy_score(
                y_valid_lgb, pred
            ),
            "tree_count_or_best_iteration": int(tree_count),})

    cross_model_results_df = (
        pd.DataFrame(cross_model_rows)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True) )

    display(cross_model_results_df.round(5))


    # Hitung manfaat Prefix8 di masing-masing algoritma.
    cross_scores = cross_model_results_df.set_index("model")["macro_f1"]

    catboost_prefix8_gain = (
        cross_scores["CatBoost SqrtBalanced + Prefix8"]
        - cross_scores["CatBoost SqrtBalanced Original"])

    lightgbm_prefix8_gain = (
        cross_scores["LightGBM Prefix8 Original"]
        - cross_scores["LightGBM Baseline Original"])

    print(
        "Macro F1 gain from Prefix8 on CatBoost:",
        round(catboost_prefix8_gain, 5))

    print(
        "Macro F1 gain from Prefix8 on LightGBM:",
        round(lightgbm_prefix8_gain, 5))

else:
    print("CatBoost Prefix8: eksperimen dilewati.")

CatBoost Prefix8: eksperimen dilewati.


Dengan demikian, LightGBM Prefix8 dipilih sebagai model utama untuk tahap eksperimen berikutnya karena memberikan Macro F1 tertinggi pada evaluasi saat ini. Selanjutnya, model ini akan digunakan untuk evaluasi lebih lanjut dan persiapan prediksi pada test set.

## 14. Evaluasi per kelas untuk Prefix8

In [ ]:
if RUN_LGBM_EXPERIMENTS:
    prefix8_pred_labels = int_to_grade[prefix8_pred.astype(int)]
    prefix8_adjusted_labels = int_to_grade[prefix8_pred_adjusted.astype(int)]

    print("=== Prefix8 Original ===")
    print(classification_report(
        y_valid, prefix8_pred_labels,
        labels=label_order, target_names=label_order,
        digits=4, zero_division=0,))

    print("=== Prefix8 Adjusted ===")
    print(classification_report(
        y_valid, prefix8_adjusted_labels,
        labels=label_order, target_names=label_order,
        digits=4, zero_division=0,))

    cm = confusion_matrix(
        y_valid, prefix8_pred_labels,
        labels=label_order, normalize="true")
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=label_order
    ).plot(ax=ax, cmap="Blues", values_format=".1%", xticks_rotation=45)
    ax.set_title("LightGBM Prefix8 Original: Row-normalized Confusion Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("Evaluasi Prefix8: eksperimen dilewati.")

Evaluasi Prefix8: eksperimen dilewati.


Perbedaan Hasil LightGBM Original dan Adjusted sangat kecil (sekitar `0.00004` pada validation pertama), sehingga model Original lebih sederhana dan dipilih untuk submission.

In [ ]:
result_tables = []

candidate_tables = [
    "classical_results_df",
    "classical_tuning_results_df",

    "tuning_results_df",
    "rf_tuning_df",
    "classical_tuning_df",
    "catboost_results_df",

    "interaction_results_df",
    "lightgbm_results_df",
    "prefix_results_df",]

for name in candidate_tables:
    df = globals().get(name)

    if isinstance(df, pd.DataFrame) and not df.empty:
        temp = df.copy()

        # Samakan nama kolom
        temp = temp.rename(columns={
            "Model": "model",
            "model_name": "model",
            "experiment": "model",
            "macro_f1_current": "macro_f1",
            "accuracy_current": "accuracy",
            "Macro F1": "macro_f1",
            "Macro F1 Score": "macro_f1",
            "f1_macro": "macro_f1",
            "Accuracy": "accuracy",})

        if {"model", "macro_f1", "accuracy"}.issubset(temp.columns):
            result_tables.append(
                temp[["model", "macro_f1", "accuracy"]])

if result_tables:
    all_results = pd.concat(
        result_tables, ignore_index=True
    ).drop_duplicates()

    display(
        all_results
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
        .round(5))
else:
    print("Tabel hasil belum ditemukan. Periksa nama variabel DataFrame.")

Tabel hasil belum ditemukan. Periksa nama variabel DataFrame.


## 16. Feature importance

In [23]:
if RUN_LGBM_EXPERIMENTS:
    feature_importance = pd.DataFrame({
        "feature": x_train_p8.columns,
        "importance": lgb_prefix8.named_steps["model"].feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    display(feature_importance.head(20))

    plot_importance = feature_importance.head(15).sort_values("importance")
    plt.figure(figsize=(9, 6))
    plt.barh(plot_importance["feature"], plot_importance["importance"])
    plt.title("LightGBM Prefix8: Top 15 Feature Importances")
    plt.xlabel("Split importance")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance: eksperimen dilewati.")

Feature importance: eksperimen dilewati.


## 17. Keputusan model final

LightGBM Prefix8 Original dipilih karena:

1. Macro F1 validation sekitar `0.5692`, lebih tinggi dari baseline LightGBM sekitar `0.551`.
2. Prefix8 mengungguli Prefix6, sedangkan Prefix10 turun.
3. Peningkatan tetap terlihat pada validation split kedua yang pernah diuji.
4. Probability adjustment hanya memberikan perbedaan yang sangat kecil pada Prefix8, jadi Original lebih sederhana.
5. Submission Prefix8 Original sebelumnya memperoleh skor leaderboard publik `0.57239`.

Sekarang model dilatih ulang pada seluruh data berlabel. Jumlah pohon final ditetapkan 200.

## 18. Final training pada seluruh data berlabel

Fitur final terdiri dari 15 fitur asli (tanpa `building_id` mentah) ditambah satu fitur kategorikal `building_id_prefix_8`. Kategori fitur biasa dan prefix dipelajari dari seluruh train lalu diterapkan konsisten pada test.

In [24]:
final_n_estimators = 200
if RUN_FINAL_TRAINING:
    start = time.perf_counter()
    final_lgb_prefix8 = train_final_lightgbm(
        train, target, feature_cols, categorical_cols,
        grade_to_int, random_state, final_n_estimators)
    final_minutes = (time.perf_counter() - start) / 60
    x_full = final_lgb_prefix8.named_steps["features"].transform(train)
    x_test_final = final_lgb_prefix8.named_steps["features"].transform(test)
    final_categorical_cols = categorical_cols + ["building_id_prefix_8"]
    print("Final training time:", round(final_minutes, 2), "minutes")
    print("Final feature count:", x_full.shape[1])
    print("Final trees:", final_lgb_prefix8.named_steps["model"].n_estimators)
    model_path = ROOT / "models" / "lightgbm_prefix8_final.pkl"
    joblib.dump(final_lgb_prefix8, model_path)
    print("Saved:", model_path)
else:
    print("Training final dilewati; jalankan 03_submission untuk memuat atau melatih model.")

Final training time: 1.43 minutes
Final feature count: 16
Final trees: 200
Saved: C:\Users\Sharliz\Downloads\last structural damage\structural-damage-classification\models\lightgbm_prefix8_final.pkl


## 20. Kesimpulan

Proses modeling dimulai dari pemeriksaan data dan EDA, dilanjutkan baseline Dummy dan model scikit-learn, kemudian CatBoost dan LightGBM. Baseline LightGBM menjadi titik awal untuk investigasi ID, bukan karena sejak awal kita menganggap ID berguna.

Audit `building_id` menemukan bahwa dua digit awal cocok dengan `district_id` dan prefix yang lebih panjang membentuk kelompok dengan ukuran yang berbeda. Prefix diuji sebagai kategori. Prefix8 paling konsisten meningkatkan Macro F1 di eksperimen yang sudah dilakukan, sementara Prefix10 menurun dan interaksi lokasi yang dicoba tidak memberikan peningkatan.

Model final notebook ini adalah **LightGBM native multiclass + `building_id` Prefix8**, dengan `class_weight="balanced"`. 